# Increasing Hidden Fraction

Grid search on `train_clamped_visible` varying `hidden_cell_fraction` from 0.1 to 0.9.

In [ ]:
import shutil
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import toml
import zarr
from torch.utils.data import DataLoader, Dataset

sys.path.insert(
    0,
    "/tachyon/groups/scratch/gzenke/bedfrory/teacher-student/hidden-activity/scripts",
)

from connectome_snns.utils.reproducibility import load_experiment_config
from _em_core import run_estep_clamped, run_estep_free
from connectome_snns.analysis import (
    fluctuation_r_squared,
    interleave_spike_trains,
    make_grid_colormap,
    r_squared,
)
from connectome_snns.configs.conductance_based import FeedforwardLayerConfig, RecurrentLayerConfig
from connectome_snns.dataloaders.supervised import CyclicSampler, ExactFFDataset, SpikeData
from connectome_snns.network_simulators.feedforward_conductance_based.simulator import (
    FeedforwardConductanceLIFNetwork,
)
from connectome_snns.network_simulators.projections import make_frozen_chunked_ff_projections
from connectome_snns.visualization import (
    EXCITATORY_COLOR,
    INHIBITORY_COLOR,
    plot_firing_rate_scatter,
    plot_r2_vs_parameter,
    plot_spike_trains,
    use_project_style,
)
from connectome_snns.visualization.scaling_factors import SF_PATHWAYS, plot_sf_trajectories

use_project_style()

In [ ]:
hidden_fraction_config = load_experiment_config("experiment.toml")
GRID_DIR = hidden_fraction_config["output_dir"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SMOOTH_WINDOW = 10  # 50 epochs / 5 epoch log interval
MAX_RATE = None  # auto-scale
TAU_MS = 50.0  # Gaussian kernel std in ms
N_NEURONS_SHOW = 10
T_SHOW_MS = 5000.0  # crop rasters to 5 seconds

In [ ]:
class SingleBatchDataset(Dataset):
    """Wraps a ExactFFDataset to only return batch index 0."""

    def __init__(self, dataset):
        self.dataset = dataset
        self.batch_size = 1
        self.dt = dataset.dt
        self.num_chunks = dataset.num_chunks

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        data = self.dataset[idx]
        return SpikeData(
            input_spikes=data.input_spikes[:1],
            target_spikes=data.target_spikes[:1],
        )


def _learned_sf_matrix(ckpt_state, ff_cell_type_names, rec_cell_type_names):
    """Reconstruct an (n_ff_ct + n_rec_ct, n_rec_ct) SF matrix from per-pair log_sf entries."""
    n_ff_ct = len(ff_cell_type_names)
    n_rec_ct = len(rec_cell_type_names)
    src_names = list(ff_cell_type_names) + list(rec_cell_type_names)
    sf = np.ones((n_ff_ct + n_rec_ct, n_rec_ct), dtype=np.float32)
    for si, src in enumerate(src_names):
        for ti, tgt in enumerate(rec_cell_type_names):
            key = f"ff_projections.{src}__{tgt}.log_sf"
            if key in ckpt_state:
                sf[si, ti] = float(torch.exp(ckpt_state[key]).item())
    return sf


def compute_per_neuron_rates(exp_dir, e_step_type, loss_type, device="cpu"):
    """Run E-step + M-step inference with final learned scaling factors (batch 0 only).

    Returns dict with hidden_rates, visible_rates, teacher_rates, indices, cell types.
    Results are cached to per_neuron_rates.npz.
    """
    cache_path = exp_dir / "per_neuron_rates.npz"
    if cache_path.exists():
        print(f"  Cached: {exp_dir.name}")
        return dict(np.load(cache_path))

    print(f"  Computing: {exp_dir.name} ({e_step_type}/{loss_type})...")
    input_dir = exp_dir / "inputs"

    # Load parameters
    with open(exp_dir / "parameters.toml") as f:
        data = toml.load(f)

    rec_cfg = RecurrentLayerConfig(**data["recurrent"])
    ff_cfg = FeedforwardLayerConfig(**data["feedforward"])
    chunk_size = data["simulation"]["chunk_size"]
    surrgrad_scale = data["hyperparameters"]["surrgrad_scale"]

    # Network structure
    ns = np.load(input_dir / "network_structure.npz")
    weights = ns["recurrent_weights"]
    ff_weights = ns["feedforward_weights"]
    cell_type_indices = ns["cell_type_indices"]
    ff_cell_type_indices = ns["feedforward_cell_type_indices"]
    rec_mask = ns["recurrent_connectivity"]
    ff_mask = ns["feedforward_connectivity"]
    n_full = weights.shape[0]
    n_ff = ff_weights.shape[0]

    # Hidden/visible indices
    hi = np.load(exp_dir / "targets" / "hidden_neurons.npz")
    hidden_indices = hi["hidden_indices"]
    visible_indices = hi["visible_indices"]

    # Cell/synapse params
    rec_cell_params = rec_cfg.get_cell_params()
    ff_cell_params = ff_cfg.get_cell_params()
    n_ff_ct = len(ff_cell_params)
    rec_syn_params = rec_cfg.get_synapse_params()
    ff_syn_params = ff_cfg.get_synapse_params()
    n_ff_st = len(ff_syn_params)

    rec_cell_type_names = [cp["name"] for cp in rec_cell_params]
    ff_cell_type_names = [cp["name"] for cp in ff_cell_params]

    cat_cell_type_indices = np.concatenate(
        [ff_cell_type_indices, cell_type_indices + n_ff_ct]
    )

    combined_cell_params = ff_cell_params.copy()
    for cp in rec_cell_params:
        o = cp.copy()
        o["cell_id"] = cp["cell_id"] + n_ff_ct
        combined_cell_params.append(o)

    combined_syn_params = ff_syn_params.copy()
    for sp in rec_syn_params:
        o = sp.copy()
        o["cell_id"] = sp["cell_id"] + n_ff_ct
        o["synapse_id"] = sp["synapse_id"] + n_ff_st
        combined_syn_params.append(o)

    # Reconstruct perturbed weights from saved target scaling factors
    tgt_sf = np.load(exp_dir / "targets" / "target_scaling_factors.npz")
    perturbation = 1.0 / tgt_sf["feedforward_scaling_factors"]

    cat_weights = np.concatenate([ff_weights, weights], axis=0)
    pert_weights = cat_weights.copy()
    for i in range(n_ff + n_full):
        for j in range(n_full):
            pert_weights[i, j] *= perturbation[
                cat_cell_type_indices[i], cell_type_indices[j]
            ]
    pert_ff = pert_weights[:n_ff, :]
    pert_rec = pert_weights[n_ff:, :]

    # Extract learned scaling factors from final checkpoint
    em_dirs = sorted(exp_dir.glob("em_iter_*"))
    ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_best.pt"
    if not ckpt_path.exists():
        ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_latest.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    learned_sf = _learned_sf_matrix(
        ckpt["model_state_dict"], ff_cell_type_names, rec_cell_type_names
    )
    sf_ff = learned_sf[:n_ff_ct]
    sf_rec = learned_sf[n_ff_ct:]

    # Spike dataset -- single batch wrapper for speed
    ds_full = ExactFFDataset(
        spike_data_path=input_dir / "spike_data.zarr",
        chunk_size=chunk_size,
        device=device,
    )
    ds = SingleBatchDataset(ds_full)
    dt_ms = ds.dt
    num_chunks = ds.num_chunks

    # === E-step: infer hidden neuron spikes ===
    estep_path = exp_dir / "_temp_inference.zarr"
    if estep_path.exists():
        shutil.rmtree(estep_path)

    if e_step_type == "clamped":
        run_estep_clamped(
            dt=dt_ms,
            batch_size=1,
            device=device,
            perturbed_rec_weights=pert_rec,
            perturbed_ff_weights=pert_ff,
            cell_type_indices=cell_type_indices,
            feedforward_cell_type_indices=ff_cell_type_indices,
            recurrent_cell_params=rec_cell_params,
            feedforward_cell_params=ff_cell_params,
            recurrent_synapse_params=rec_syn_params,
            feedforward_synapse_params=ff_syn_params,
            surrgrad_scale=surrgrad_scale,
            sf_recurrent=sf_rec,
            sf_feedforward=sf_ff,
            recurrent_mask=rec_mask,
            feedforward_mask=ff_mask,
            visible_indices=visible_indices,
            hidden_indices=hidden_indices,
            spike_dataset=ds,
            num_chunks=num_chunks,
            chunk_size=chunk_size,
            output_path=estep_path,
            n_ff_cell_types=n_ff_ct,
        )
    else:
        run_estep_free(
            dt=dt_ms,
            batch_size=1,
            device=device,
            perturbed_rec_weights=pert_rec,
            perturbed_ff_weights=pert_ff,
            cell_type_indices=cell_type_indices,
            feedforward_cell_type_indices=ff_cell_type_indices,
            recurrent_cell_params=rec_cell_params,
            feedforward_cell_params=ff_cell_params,
            recurrent_synapse_params=rec_syn_params,
            feedforward_synapse_params=ff_syn_params,
            surrgrad_scale=surrgrad_scale,
            sf_recurrent=sf_rec,
            sf_feedforward=sf_ff,
            recurrent_mask=rec_mask,
            feedforward_mask=ff_mask,
            hidden_indices=hidden_indices,
            spike_dataset=ds,
            num_chunks=num_chunks,
            chunk_size=chunk_size,
            output_path=estep_path,
        )

    estep_zarr = zarr.open_group(estep_path, mode="r")
    hidden_spikes_zarr = estep_zarr["output_spikes"]

    # === M-step: simulate visible neuron spikes ===
    if loss_type == "visible":
        model_ff_block = pert_ff[:, visible_indices] * ff_mask[
            :, visible_indices
        ].astype(np.float32)
        model_rec_block = pert_rec[:, visible_indices] * rec_mask[
            :, visible_indices
        ].astype(np.float32)
        model_ct = cell_type_indices[visible_indices]
    else:
        model_ff_block = pert_ff * ff_mask.astype(np.float32)
        model_rec_block = pert_rec * rec_mask.astype(np.float32)
        model_ct = cell_type_indices

    [cp["name"] for cp in combined_cell_params]
    mstep_projections = make_frozen_chunked_ff_projections(
        rec_weights=model_rec_block,
        ff_weights=model_ff_block,
        cell_type_indices=model_ct,
        ff_cell_type_indices=ff_cell_type_indices,
        cell_type_names=rec_cell_type_names,
        ff_cell_type_names=ff_cell_type_names,
        scaling_factors=np.concatenate([sf_ff, sf_rec], axis=0),
    )

    model = FeedforwardConductanceLIFNetwork(
        dt=dt_ms,
        projections=mstep_projections,
        cell_type_indices=model_ct,
        cell_type_indices_FF=cat_cell_type_indices,
        cell_params=rec_cell_params,
        cell_params_FF=combined_cell_params,
        synapse_params_FF=combined_syn_params,
        surrgrad_scale=surrgrad_scale,
        batch_size=1,
        track_variables=False,
    )
    model.to(device)
    model.eval()
    model.reset_state(batch_size=1)

    vis_t = torch.from_numpy(visible_indices).long()
    hid_t = torch.from_numpy(hidden_indices).long()
    dl = DataLoader(ds, batch_size=None, sampler=CyclicSampler(ds), num_workers=0)

    all_vis_spikes = []
    with torch.inference_mode():
        for ci, batch in enumerate(dl):
            if ci >= num_chunks:
                break
            _, ts, _ = batch.target_spikes.shape
            rec_input = torch.zeros(1, ts, n_full, device=device, dtype=torch.float32)
            rec_input[:, :, vis_t] = batch.target_spikes[:, :, vis_t].float()
            hid_chunk = np.array(hidden_spikes_zarr[:1, ci * ts : (ci + 1) * ts, :])
            rec_input[:, :, hid_t] = torch.from_numpy(hid_chunk).float().to(device)
            model_input = torch.cat([batch.input_spikes, rec_input], dim=2)
            out = model.forward(input_spikes=model_input)
            all_vis_spikes.append(out[0].bool().cpu().numpy())

    vis_spikes = np.concatenate(all_vis_spikes, axis=0)
    hid_spikes = np.array(hidden_spikes_zarr[0, :, :])

    duration_s = num_chunks * chunk_size * dt_ms / 1000.0
    hidden_rates = hid_spikes.sum(axis=0) / duration_s
    if loss_type == "visible":
        visible_rates = vis_spikes.sum(axis=0) / duration_s
    else:
        visible_rates = vis_spikes[:, visible_indices].sum(axis=0) / duration_s

    # Teacher rates
    teacher_zarr = zarr.open_group(input_dir / "spike_data.zarr", mode="r")
    teacher_sp = np.array(teacher_zarr["output_spikes"][0, :, :])
    teacher_rates = teacher_sp.sum(axis=0) / (teacher_sp.shape[0] * dt_ms / 1000.0)

    result = dict(
        hidden_rates=hidden_rates,
        visible_rates=visible_rates,
        teacher_rates=teacher_rates,
        hidden_indices=hidden_indices,
        visible_indices=visible_indices,
        cell_type_indices=cell_type_indices,
    )
    np.savez(cache_path, **result)

    # Cleanup
    shutil.rmtree(estep_path)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    print(
        f"    Done: {len(hidden_indices)} hidden, {len(visible_indices)} visible neurons"
    )
    return result


def compute_spike_rasters(exp_dir, device="cpu"):
    """Load teacher + student spikes for raster plotting. Cached to spike_rasters_full.npz.

    Uses plot_size from experiment parameters for the number of chunks.
    """
    cache_path = exp_dir / "spike_rasters_full.npz"
    if cache_path.exists():
        print(f"  Cached rasters: {exp_dir.name}")
        return dict(np.load(cache_path))

    print(f"  Computing rasters: {exp_dir.name}...")
    input_dir = exp_dir / "inputs"

    with open(exp_dir / "parameters.toml") as f:
        data = toml.load(f)

    rec_cfg = RecurrentLayerConfig(**data["recurrent"])
    ff_cfg = FeedforwardLayerConfig(**data["feedforward"])
    chunk_size = data["simulation"]["chunk_size"]
    surrgrad_scale = data["hyperparameters"]["surrgrad_scale"]
    n_chunks_raster = data["training"]["plot_size"]

    ns = np.load(input_dir / "network_structure.npz")
    weights = ns["recurrent_weights"]
    ff_weights = ns["feedforward_weights"]
    cell_type_indices = ns["cell_type_indices"]
    ff_cell_type_indices = ns["feedforward_cell_type_indices"]
    rec_mask = ns["recurrent_connectivity"]
    ff_mask = ns["feedforward_connectivity"]
    n_full = weights.shape[0]
    n_ff = ff_weights.shape[0]

    hi = np.load(exp_dir / "targets" / "hidden_neurons.npz")
    hidden_indices = hi["hidden_indices"]
    visible_indices = hi["visible_indices"]

    rec_cell_params = rec_cfg.get_cell_params()
    ff_cell_params = ff_cfg.get_cell_params()
    n_ff_ct = len(ff_cell_params)
    rec_syn_params = rec_cfg.get_synapse_params()
    ff_syn_params = ff_cfg.get_synapse_params()
    n_ff_st = len(ff_syn_params)

    rec_cell_type_names = [cp["name"] for cp in rec_cell_params]
    ff_cell_type_names = [cp["name"] for cp in ff_cell_params]

    cat_cell_type_indices = np.concatenate(
        [ff_cell_type_indices, cell_type_indices + n_ff_ct]
    )

    combined_cell_params = ff_cell_params.copy()
    for cp in rec_cell_params:
        o = cp.copy()
        o["cell_id"] = cp["cell_id"] + n_ff_ct
        combined_cell_params.append(o)

    combined_syn_params = ff_syn_params.copy()
    for sp in rec_syn_params:
        o = sp.copy()
        o["cell_id"] = sp["cell_id"] + n_ff_ct
        o["synapse_id"] = sp["synapse_id"] + n_ff_st
        combined_syn_params.append(o)

    # Perturbed weights
    tgt_sf = np.load(exp_dir / "targets" / "target_scaling_factors.npz")
    perturbation = 1.0 / tgt_sf["feedforward_scaling_factors"]
    cat_weights = np.concatenate([ff_weights, weights], axis=0)
    pert_weights = cat_weights.copy()
    for i in range(n_ff + n_full):
        for j in range(n_full):
            pert_weights[i, j] *= perturbation[
                cat_cell_type_indices[i], cell_type_indices[j]
            ]
    pert_ff = pert_weights[:n_ff, :]
    pert_rec = pert_weights[n_ff:, :]

    # Teacher spikes
    t_end = n_chunks_raster * chunk_size
    teacher_zarr = zarr.open_group(input_dir / "spike_data.zarr", mode="r")
    teacher_all = np.array(teacher_zarr["output_spikes"][0, :t_end, :])
    dt_ms = float(teacher_zarr.attrs.get("dt", 1.0))

    # Hidden student spikes from last EM iteration
    em_dirs = sorted(exp_dir.glob("em_iter_*"))
    inferred_zarr = zarr.open_group(em_dirs[-1] / "inferred_spikes.zarr", mode="r")
    hidden_student = np.array(inferred_zarr["output_spikes"][0, :t_end, :])

    # Learned scaling factors from final checkpoint
    ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_best.pt"
    if not ckpt_path.exists():
        ckpt_path = em_dirs[-1] / "checkpoints" / "checkpoint_latest.pt"
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    learned_sf = _learned_sf_matrix(
        ckpt["model_state_dict"], ff_cell_type_names, rec_cell_type_names
    )
    sf_ff = learned_sf[:n_ff_ct]
    sf_rec = learned_sf[n_ff_ct:]

    # M-step model for visible student spikes
    model_ff_block = pert_ff[:, visible_indices] * ff_mask[:, visible_indices].astype(
        np.float32
    )
    model_rec_block = pert_rec[:, visible_indices] * rec_mask[
        :, visible_indices
    ].astype(np.float32)
    model_ct = cell_type_indices[visible_indices]

    mstep_projections = make_frozen_chunked_ff_projections(
        rec_weights=model_rec_block,
        ff_weights=model_ff_block,
        cell_type_indices=model_ct,
        ff_cell_type_indices=ff_cell_type_indices,
        cell_type_names=rec_cell_type_names,
        ff_cell_type_names=ff_cell_type_names,
        scaling_factors=np.concatenate([sf_ff, sf_rec], axis=0),
    )

    model = FeedforwardConductanceLIFNetwork(
        dt=dt_ms,
        projections=mstep_projections,
        cell_type_indices=model_ct,
        cell_type_indices_FF=cat_cell_type_indices,
        cell_params=rec_cell_params,
        cell_params_FF=combined_cell_params,
        synapse_params_FF=combined_syn_params,
        surrgrad_scale=surrgrad_scale,
        batch_size=1,
        track_variables=False,
    )
    model.to(device)
    model.eval()
    model.reset_state(batch_size=1)

    ds_full = ExactFFDataset(
        spike_data_path=input_dir / "spike_data.zarr",
        chunk_size=chunk_size,
        device=device,
    )
    ds = SingleBatchDataset(ds_full)
    vis_t = torch.from_numpy(visible_indices).long()
    hid_t = torch.from_numpy(hidden_indices).long()
    dl = DataLoader(ds, batch_size=None, sampler=CyclicSampler(ds), num_workers=0)

    all_vis_spikes = []
    with torch.inference_mode():
        for ci, batch in enumerate(dl):
            if ci >= n_chunks_raster:
                break
            _, ts, _ = batch.target_spikes.shape
            rec_input = torch.zeros(1, ts, n_full, device=device, dtype=torch.float32)
            rec_input[:, :, vis_t] = batch.target_spikes[:, :, vis_t].float()
            hid_chunk = np.array(
                inferred_zarr["output_spikes"][:1, ci * ts : (ci + 1) * ts, :]
            )
            rec_input[:, :, hid_t] = torch.from_numpy(hid_chunk).float().to(device)
            model_input = torch.cat([batch.input_spikes, rec_input], dim=2)
            out = model.forward(input_spikes=model_input)
            all_vis_spikes.append(out[0].cpu().numpy())

    visible_student = np.concatenate(all_vis_spikes, axis=0)

    result = dict(
        teacher_visible=teacher_all[:, visible_indices].astype(bool),
        teacher_hidden=teacher_all[:, hidden_indices].astype(bool),
        student_visible=visible_student.astype(bool),
        student_hidden=hidden_student.astype(bool),
        visible_indices=visible_indices,
        hidden_indices=hidden_indices,
    )
    np.savez(cache_path, **result)
    del model
    if device == "cuda":
        torch.cuda.empty_cache()
    return result

## Load Grid Search Data

In [ ]:
fractions = np.arange(0.1, 1.0, 0.1).round(1)
grid_metrics = {}
for frac in fractions:
    csv_path = GRID_DIR / f"hidden-frac-{frac}" / "training_metrics.csv"
    if csv_path.exists():
        grid_metrics[frac] = pd.read_csv(csv_path)
        print(f"hidden_cell_fraction={frac}: {len(grid_metrics[frac])} rows")

sorted_fracs = sorted(grid_metrics.keys())
norm, color_dict = make_grid_colormap(sorted_fracs)

## Scaling Factor Convergence by Hidden Fraction

In [ ]:
fig = plot_sf_trajectories(
    grid_metrics,
    x_scale=1 / 50,
    pathways=SF_PATHWAYS,
    colors={f: color_dict[f] for f in sorted_fracs},
    label_fn=lambda f: f"{f:.1f}",
    suptitle="Scaling Factor Convergence vs Hidden Cell Fraction",
    xlabel="EM Iteration",
    figsize=(12, 10),
)
# Add legend to first axis only
axes = fig.axes
if axes:
    axes[0].legend(title="Hidden Frac.", fontsize=7, loc="best", ncol=3)
plt.show()

## Final Scaling Factors vs Hidden Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: individual scaling factors
ax = axes[0]
for key, title in SF_PATHWAYS:
    col = f"scaling_factors/{key}_value"
    values = [grid_metrics[f].iloc[-1][col] for f in sorted_fracs]
    ax.plot(sorted_fracs, values, "o-", label=title, markersize=5)

ax.axhline(y=1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax.set_xlabel("Hidden Cell Fraction")
ax.set_ylabel("Final Scaling Factor")
ax.set_ylim([0, 1.5])
ax.set_title("Individual Scaling Factors")
ax.legend(fontsize=7)

# Right: mean absolute deviation from target
ax = axes[1]
mean_devs = []
for frac in sorted_fracs:
    final = grid_metrics[frac].iloc[-1]
    sf_values = np.array([final[f"scaling_factors/{k}_value"] for k, _ in SF_PATHWAYS])
    mean_devs.append(np.mean(np.abs(sf_values - 1.0)))

ax.plot(sorted_fracs, mean_devs, "ko-", markersize=6)
ax.set_xlabel("Hidden Cell Fraction")
ax.set_ylabel("Mean |SF - target|")
ax.set_title("Mean Scaling Factor Error")

plt.suptitle(
    "Final Scaling Factors vs Hidden Cell Fraction", fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.show()

# Save left panel only
fig_left, ax_left = plt.subplots(figsize=(6, 4))
for key, title in SF_PATHWAYS:
    col = f"scaling_factors/{key}_value"
    values = [grid_metrics[f].iloc[-1][col] for f in sorted_fracs]
    ax_left.plot(sorted_fracs, values, "o-", label=title, markersize=5)

ax_left.axhline(y=1.0, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax_left.set_xlabel("Hidden Cell Fraction")
ax_left.set_ylabel("Final Scaling Factor")
ax_left.set_ylim([0, 1.5])
ax_left.set_title("Final Scaling Factors vs Hidden Cell Fraction", fontweight="bold")
ax_left.legend(fontsize=7)
fig_left.tight_layout()
plt.show()

## Loss vs Hidden Fraction

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: loss trajectories coloured by fraction
ax = axes[0]
for frac, df in sorted(grid_metrics.items()):
    smoothed = df["van_rossum_loss"].rolling(SMOOTH_WINDOW).mean()
    ax.plot(
        df["epoch"] / 50,
        smoothed,
        color=color_dict[frac],
        linewidth=1.2,
        label=f"{frac:.1f}",
    )

ax.set_xlabel("EM Iteration")
ax.set_ylabel("Van Rossum Loss")
ax.set_title("Loss Trajectories")
ax.set_ylim(0, None)
ax.legend(title="Hidden Frac.", fontsize=7, ncol=3)

# Right: final loss vs fraction
ax = axes[1]
final_losses = [grid_metrics[f].iloc[-1]["van_rossum_loss"] for f in sorted_fracs]
ax.plot(sorted_fracs, final_losses, "ko-", markersize=6)
ax.set_xlabel("Hidden Cell Fraction")
ax.set_ylabel("Final Van Rossum Loss")
ax.set_title("Final Loss")

plt.suptitle("Loss vs Hidden Cell Fraction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Firing Rates vs Hidden Fraction

In [ ]:
fr_grid = [
    (0, 0, "visible", "excitatory", "Visible Excitatory"),
    (0, 1, "visible", "inhibitory", "Visible Inhibitory"),
    (1, 0, "hidden", "excitatory", "Hidden Excitatory"),
    (1, 1, "hidden", "inhibitory", "Hidden Inhibitory"),
]

for stat, stat_label, suptitle in [
    (
        "mean",
        "Mean Firing Rate (Hz)",
        "Mean Population Firing Rates vs Hidden Cell Fraction",
    ),
    ("std", "Std Firing Rate (Hz)", "Firing Rate Std vs Hidden Cell Fraction"),
]:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)

    for row, col, visibility, cell_type, title in fr_grid:
        ax = axes[row, col]
        student_col = f"firing_rate/student_{visibility}_{cell_type}_{stat}"
        teacher_col = f"firing_rate/teacher_{visibility}_{cell_type}_{stat}"

        for frac, df in sorted(grid_metrics.items()):
            if student_col not in df.columns:
                continue
            x = df["epoch"] / 50
            y = df[student_col].rolling(SMOOTH_WINDOW).mean()
            ax.plot(
                x,
                y,
                color=color_dict[frac],
                linewidth=1.2,
                label=f"{frac:.1f}" if row == 0 and col == 0 else None,
            )

        first_df = next(iter(grid_metrics.values()))
        if teacher_col in first_df.columns:
            x = first_df["epoch"] / 50
            t_y = first_df[teacher_col].rolling(SMOOTH_WINDOW).mean()
            ax.plot(
                x,
                t_y,
                color="black",
                linestyle="--",
                linewidth=1.5,
                label="Teacher" if row == 0 and col == 0 else None,
            )

        ax.set_title(title)
        ax.set_ylim(0, None)
        if row == 1:
            ax.set_xlabel("EM Iteration")
        if col == 0:
            ax.set_ylabel(stat_label)
        if row == 0 and col == 0:
            ax.legend(title="Hidden Frac.", fontsize=7, ncol=3)

    plt.suptitle(suptitle, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

## Per-Neuron Firing Rates (Inference with Final Scaling Factors)

In [ ]:
GROUP_NAMES = [
    "Visible Excitatory",
    "Visible Inhibitory",
    "Hidden Excitatory",
    "Hidden Inhibitory",
]
GROUP_COLORS = [EXCITATORY_COLOR, INHIBITORY_COLOR, EXCITATORY_COLOR, INHIBITORY_COLOR]

# First pass: compute all rates and find global axis limit
all_rates = {}
global_max = 0.0

for frac in sorted_fracs:
    exp_dir = GRID_DIR / f"hidden-frac-{frac}"
    rates = compute_per_neuron_rates(exp_dir, "clamped", "visible", DEVICE)
    all_rates[frac] = rates
    for idx, student in [
        (rates["visible_indices"], rates["visible_rates"]),
        (rates["hidden_indices"], rates["hidden_rates"]),
    ]:
        teacher = rates["teacher_rates"][idx]
        global_max = max(global_max, teacher.max(), student.max())

global_max *= 1.05

# Second pass: plot scatter per fraction
for frac in sorted_fracs:
    rates = all_rates[frac]
    vis_idx = rates["visible_indices"]
    hid_idx = rates["hidden_indices"]

    teacher = np.concatenate(
        [rates["teacher_rates"][vis_idx], rates["teacher_rates"][hid_idx]]
    )
    student = np.concatenate([rates["visible_rates"], rates["hidden_rates"]])
    group = np.concatenate(
        [
            rates["cell_type_indices"][vis_idx],  # 0=E, 1=I for visible
            rates["cell_type_indices"][hid_idx] + 2,  # 2=E, 3=I for hidden
        ]
    )

    fig, axes = plt.subplots(2, 2, figsize=(8, 8))
    plot_firing_rate_scatter(
        teacher,
        student,
        group,
        split_panels=True,
        axes=[axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]],
        cell_type_names=GROUP_NAMES,
        cell_type_colors=GROUP_COLORS,
        max_rate=global_max,
        marker_size=4,
        alpha=0.3,
    )
    fig.suptitle(f"Hidden Fraction = {frac}", fontsize=14, fontweight="bold")
    fig.tight_layout()
    if frac == 0.9:
        plt.show()

## Spike Raster Comparisons

Teacher vs student spike trains for visible and hidden neurons, plotted separately for each hidden fraction.

In [ ]:
# Compute rasters for all fractions (uses plot_size chunks, runs on GPU)
all_rasters = {}
for frac in sorted_fracs:
    exp_dir = GRID_DIR / f"hidden-frac-{frac}"
    all_rasters[frac] = compute_spike_rasters(exp_dir, DEVICE)

# Load dt
first_exp_dir = GRID_DIR / f"hidden-frac-{sorted_fracs[0]}"
_zarr = zarr.open_group(first_exp_dir / "inputs" / "spike_data.zarr", mode="r")
raster_dt = float(_zarr.attrs.get("dt", 1.0))
del _zarr

# Find neurons that are always-visible / always-hidden across ALL fractions
all_visible_sets = [set(all_rasters[f]["visible_indices"]) for f in sorted_fracs]
all_hidden_sets = [set(all_rasters[f]["hidden_indices"]) for f in sorted_fracs]
always_visible = np.array(sorted(set.intersection(*all_visible_sets)))
always_hidden = np.array(sorted(set.intersection(*all_hidden_sets)))

print(f"Always-visible neurons: {len(always_visible)}")
print(f"Always-hidden neurons:  {len(always_hidden)}")

rng = np.random.RandomState(42)
show_visible_ids = np.sort(rng.choice(always_visible, N_NEURONS_SHOW, replace=False))
show_hidden_ids = np.sort(rng.choice(always_hidden, N_NEURONS_SHOW, replace=False))

# Crop rasters to T_SHOW_MS
n_crop = int(T_SHOW_MS / raster_dt)

for frac in sorted_fracs:
    rasters = all_rasters[frac]
    vis_idx = rasters["visible_indices"]
    hid_idx = rasters["hidden_indices"]

    vis_id_to_col = {int(nid): col for col, nid in enumerate(vis_idx)}
    hid_id_to_col = {int(nid): col for col, nid in enumerate(hid_idx)}

    vis_cols = [vis_id_to_col[nid] for nid in show_visible_ids]
    hid_cols = [hid_id_to_col[nid] for nid in show_hidden_ids]

    for vis_type, cols in [("visible", vis_cols), ("hidden", hid_cols)]:
        teacher = rasters[f"teacher_{vis_type}"][:n_crop, cols]
        student = rasters[f"student_{vis_type}"][:n_crop, cols]

        interleaved, ct_idx = interleave_spike_trains(
            teacher, student, n_neurons=N_NEURONS_SHOW
        )

        fig = plot_spike_trains(
            spikes=interleaved,
            dt=raster_dt,
            cell_type_indices=ct_idx,
            cell_type_names=["Teacher", "Student"],
            n_neurons_plot=2 * N_NEURONS_SHOW,
            n_compared=2,
            fraction=1.0,
            random_seed=None,
            title=f"{vis_type.title()} Neurons \u2014 Hidden Fraction = {frac}",
            figsize=(16, 6),
        )
        if frac == 0.9 and vis_type == "hidden":
            plt.show()

## Summary

In [ ]:
rows = []
for frac in sorted_fracs:
    final = grid_metrics[frac].iloc[-1]
    sf_values = np.array([final[f"scaling_factors/{k}_value"] for k, _ in SF_PATHWAYS])

    rows.append(
        {
            "Hidden Frac.": frac,
            "Final Loss": f"{final['van_rossum_loss']:.2f}",
            "FR Vis Exc (Hz)": f"{final['firing_rate/student_visible_excitatory_mean']:.2f}",
            "FR Vis Inh (Hz)": f"{final['firing_rate/student_visible_inhibitory_mean']:.2f}",
            "SF Range": f"[{sf_values.min():.3f}, {sf_values.max():.3f}]",
            "Mean |SF - 1|": f"{np.mean(np.abs(sf_values - 1.0)):.3f}",
        }
    )

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Activity R² vs Hidden Fraction

Per-neuron firing rate R² between teacher and student, broken down by visibility (visible / hidden) and cell type, as a function of the hidden cell fraction.

In [ ]:
r2_activity_visible = []
r2_activity_hidden = []

for frac in sorted_fracs:
    rates = all_rates[frac]

    for idx, student, out_list in [
        (rates["visible_indices"], rates["visible_rates"], r2_activity_visible),
        (rates["hidden_indices"], rates["hidden_rates"], r2_activity_hidden),
    ]:
        t = rates["teacher_rates"][idx]
        out_list.append(r_squared(t, student))

plot_r2_vs_parameter(
    sorted_fracs,
    {"Visible": r2_activity_visible, "Hidden": r2_activity_hidden},
    xlabel="Hidden Cell Fraction",
    title="Firing Rate R\u00b2 vs Hidden Fraction",
)
plt.show()

## Spike Train Fluctuation Matching (Gaussian-Smoothed R²)

Measure how well the student matches the *temporal fluctuations* of the teacher. Each neuron's spike train is convolved with a Gaussian kernel (σ = 50 ms), and the coefficient of determination (R²) is computed separately for visible and hidden neurons across all neurons × time.

In [ ]:
r2_fluct_visible = []
r2_fluct_hidden = []

for frac in sorted_fracs:
    exp_dir = GRID_DIR / f"hidden-frac-{frac}"
    cache_path = exp_dir / "fluct_r2.npz"

    if cache_path.exists():
        cached = np.load(cache_path)
        r2_vis = float(cached["r2_visible"])
        r2_hid = float(cached["r2_hidden"])
        print(
            f"frac={frac:.1f}  R\u00b2(visible)={r2_vis:.4f}  R\u00b2(hidden)={r2_hid:.4f}  (cached)"
        )
    else:
        rasters = all_rasters[frac]

        r2_vis = fluctuation_r_squared(
            rasters["teacher_visible"].astype(np.float32),
            rasters["student_visible"].astype(np.float32),
            TAU_MS,
            raster_dt,
        )
        r2_hid = fluctuation_r_squared(
            rasters["teacher_hidden"].astype(np.float32),
            rasters["student_hidden"].astype(np.float32),
            TAU_MS,
            raster_dt,
        )

        np.savez(cache_path, r2_visible=r2_vis, r2_hidden=r2_hid)
        print(
            f"frac={frac:.1f}  R\u00b2(visible)={r2_vis:.4f}  R\u00b2(hidden)={r2_hid:.4f}"
        )

    r2_fluct_visible.append(r2_vis)
    r2_fluct_hidden.append(r2_hid)

plot_r2_vs_parameter(
    sorted_fracs,
    {"Visible": r2_fluct_visible, "Hidden": r2_fluct_hidden},
    xlabel="Hidden Cell Fraction",
    title=f"Spike Train Fluctuation R\u00b2 vs Hidden Fraction (Gaussian \u03c3 = {TAU_MS:.0f} ms)",
)
plt.show()